# Modeling Experiments
Tree-based models, historical-pattern features, lagged features, and LSTM comparison experiments extracted from the original local notebook.

> This notebook is an experiment log. Final team submission results are summarized separately in the repository README.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

df = pd.read_csv('factory_hourly_final.csv')
df['localtime'] = pd.to_datetime(df['localtime'])
df['hour'] = df['localtime'].dt.hour
df['dayofweek'] = df['localtime'].dt.dayofweek
df['month'] = df['localtime'].dt.month


In [ ]:
def smape(a, f):
    a, f = np.asarray(a), np.asarray(f)
    denom = np.abs(a) + np.abs(f)
    mask = denom != 0
    return 100 * np.mean(2 * np.abs(f[mask] - a[mask]) / denom[mask])

# Example time-based validation split
split_date = df['localtime'].max() - pd.Timedelta(days=7)
train_df = df[df['localtime'] < split_date].copy()
test_df = df[df['localtime'] >= split_date].copy()

## 1. XGBoost / LightGBM baseline experiments

In [ ]:
exclude = ['module(equipment)', 'localtime', 'hourly_pow']
features = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

X_train, y_train = train_df[features].fillna(0), train_df['hourly_pow']
X_test, y_test = test_df[features].fillna(0), test_df['hourly_pow']

xgb_model = XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=7,
    subsample=0.8, colsample_bytree=0.8, tree_method='hist', random_state=42
)
lgbm_model = LGBMRegressor(
    n_estimators=1000, learning_rate=0.05, num_leaves=31, random_state=42
)

xgb_model.fit(X_train, y_train)
lgbm_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
lgbm_pred = lgbm_model.predict(X_test)

print('XGBoost SMAPE:', smape(y_test, xgb_pred))
print('LightGBM SMAPE:', smape(y_test, lgbm_pred))

## 2. Historical-pattern features

In [ ]:
# Use only historical period to build module-hour profiles
stable = df[df['month'].isin([12,1,2,3])].copy()
hist_profile = (
    stable.groupby(['module(equipment)','hour'])['activePower']
          .mean()
          .rename('historical_active_mean')
          .reset_index()
)

df_hist = df.merge(hist_profile, on=['module(equipment)','hour'], how='left')
df_hist[['module(equipment)','localtime','historical_active_mean']].head()

## 3. Lagged sensor features

In [ ]:
df_lag = df.sort_values(['module(equipment)','localtime']).copy()
lag_cols = ['activePower','std_activePow','current_change_acc','total_pf']
for col in lag_cols:
    if col in df_lag.columns:
        df_lag[f'{col}_lag1'] = df_lag.groupby('module(equipment)')[col].shift(1)

df_lag[[c for c in df_lag.columns if c.endswith('_lag1')]].head()

## 4. CatBoost comparison

In [ ]:
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(
    iterations=1500, learning_rate=0.03, depth=8,
    loss_function='RMSE', verbose=False, random_seed=42
)
cat_model.fit(X_train, y_train)
cat_pred = cat_model.predict(X_test)

print('CatBoost SMAPE:', smape(y_test, cat_pred))
print('CatBoost MAE:', mean_absolute_error(y_test, cat_pred))
print('CatBoost RMSE:', mean_squared_error(y_test, cat_pred)**0.5)

## 5. LSTM comparison experiment
The original project notes identify the LSTM experiment as Kim Jiwon's individual modeling task. The final team model used tree-based models instead.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

class PowerLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

# Sequence construction, scaling, and training were tested in the original local notebook.
# Recorded test performance in the team notes:
# MAE   8.8554
# RMSE 13.4537
# SMAPE 1.8050%